Variational Quantum Regressor + Parameterized Quantum Circuit

In [3]:
pip install numpy pandas qiskit qiskit-aer qiskit-machine-learning qiskit-algorithms

  Using cached qiskit_aer-0.17.2-cp314-cp314-win_amd64.whl.metadata (8.4 kB)
  Using cached qiskit_algorithms-0.4.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached qiskit_aer-0.17.2-cp314-cp314-win_amd64.whl (9.7 MB)
Using cached qiskit_algorithms-0.4.0-py3-none-any.whl (327 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)

   ---------------------------------------- 0/4 [setuptools]
   ---------------------------------------- 0/4 [setuptools]
   ---------------------------------------- 0/4 [setuptools]
   ---------------------------------------- 0/4 [setuptools]
   ---------------------------------------- 0/4 [setuptools]
   ---------------------------------------- 0/4 [setuptools]
   ---------------------------------------- 0/4 [setuptools]
   ---------------------------------------- 0/4 [setuptools]
   ---------------------------------------- 0/4 [setuptools]
   ---------------------------------------- 0

Mthod 1

In [4]:
import numpy as np
import pandas as pd

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_machine_learning.algorithms import VQR
from qiskit_algorithms.optimizers import COBYLA

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

In [6]:
train_df = pd.read_csv("train_angle_encoded.csv")
test_df = pd.read_csv("test_angle_encoded.csv")

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print(train_df.head())

Training shape: (3638, 7)
Testing shape: (910, 7)
   blue_angle  green_angle  log_blue_angle  log_green_angle  bg_ratio_angle  \
0    0.391552     0.316079        1.567331         1.427842        1.415845   
1    0.261544     0.160439        1.297008         1.004915        1.978384   
2    0.227513     0.192664        1.208279         1.112596        1.280255   
3    0.303988     0.213576        1.395553         1.175474        1.700027   
4    0.259250     0.207063        1.291323         1.156411        1.405343   

   stumpf_angle  depth  
0      1.711960   16.0  
1      2.241289   23.0  
2      1.598169    7.0  
3      1.977602   19.0  
4      1.717759   24.0  


In [9]:
angle_features = [
    "blue_angle",
    "green_angle",
    "log_blue_angle",
    "log_green_angle",
    "bg_ratio_angle",
    "stumpf_angle"
]

X_train = train_df[angle_features].to_numpy(dtype=float)
y_train = train_df["depth"].to_numpy(dtype=float)

X_test = test_df[angle_features].to_numpy(dtype=float)
y_test = test_df["depth"].to_numpy(dtype=float)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (3638, 6)
y_train: (3638,)
X_test: (910, 6)
y_test: (910,)


In [11]:
n_qubits = 6

In [10]:
feature_map = QuantumCircuit(n_qubits)

input_params = ParameterVector(
    "x",
    n_qubits
)

for i in range(n_qubits):
    feature_map.ry(input_params[i], i)

In [12]:
print(feature_map.draw())

     ┌──────────┐
q_0: ┤ Ry(x[0]) ├
     ├──────────┤
q_1: ┤ Ry(x[1]) ├
     ├──────────┤
q_2: ┤ Ry(x[2]) ├
     ├──────────┤
q_3: ┤ Ry(x[3]) ├
     ├──────────┤
q_4: ┤ Ry(x[4]) ├
     ├──────────┤
q_5: ┤ Ry(x[5]) ├
     └──────────┘


In [13]:
n_layers = 2

ansatz = QuantumCircuit(n_qubits)

weight_params = ParameterVector(
    "θ",
    n_qubits * 2 * n_layers
)

p = 0

for layer in range(n_layers):

    # Single-qubit trainable rotations
    for q in range(n_qubits):

        ansatz.ry(weight_params[p], q)
        p += 1

        ansatz.rz(weight_params[p], q)
        p += 1

    # Linear entanglement
    for q in range(n_qubits - 1):
        ansatz.cx(q, q + 1)

In [14]:
print(ansatz.draw())

      ┌──────────┐ ┌──────────┐     ┌───────────┐┌───────────┐             »
q_0: ─┤ Ry(θ[0]) ├─┤ Rz(θ[1]) ├──■──┤ Ry(θ[12]) ├┤ Rz(θ[13]) ├─────────────»
      ├──────────┤ ├──────────┤┌─┴─┐└───────────┘├───────────┤┌───────────┐»
q_1: ─┤ Ry(θ[2]) ├─┤ Rz(θ[3]) ├┤ X ├──────■──────┤ Ry(θ[14]) ├┤ Rz(θ[15]) ├»
      ├──────────┤ ├──────────┤└───┘    ┌─┴─┐    └───────────┘├───────────┤»
q_2: ─┤ Ry(θ[4]) ├─┤ Rz(θ[5]) ├─────────┤ X ├──────────■──────┤ Ry(θ[16]) ├»
      ├──────────┤ ├──────────┤         └───┘        ┌─┴─┐    └───────────┘»
q_3: ─┤ Ry(θ[6]) ├─┤ Rz(θ[7]) ├──────────────────────┤ X ├──────────■──────»
      ├──────────┤ ├──────────┤                      └───┘        ┌─┴─┐    »
q_4: ─┤ Ry(θ[8]) ├─┤ Rz(θ[9]) ├───────────────────────────────────┤ X ├────»
     ┌┴──────────┤┌┴──────────┤                                   └───┘    »
q_5: ┤ Ry(θ[10]) ├┤ Rz(θ[11]) ├────────────────────────────────────────────»
     └───────────┘└───────────┘                                            »

In [15]:
print("Number of trainable parameters:", ansatz.num_parameters)

Number of trainable parameters: 24


In [17]:
circuit = QuantumCircuit(n_qubits)

circuit.compose(
    feature_map,
    inplace=True
)

circuit.compose(
    ansatz,
    inplace=True
)

print(circuit.draw())

     ┌──────────┐ ┌──────────┐ ┌──────────┐     ┌───────────┐┌───────────┐»
q_0: ┤ Ry(x[0]) ├─┤ Ry(θ[0]) ├─┤ Rz(θ[1]) ├──■──┤ Ry(θ[12]) ├┤ Rz(θ[13]) ├»
     ├──────────┤ ├──────────┤ ├──────────┤┌─┴─┐└───────────┘├───────────┤»
q_1: ┤ Ry(x[1]) ├─┤ Ry(θ[2]) ├─┤ Rz(θ[3]) ├┤ X ├──────■──────┤ Ry(θ[14]) ├»
     ├──────────┤ ├──────────┤ ├──────────┤└───┘    ┌─┴─┐    └───────────┘»
q_2: ┤ Ry(x[2]) ├─┤ Ry(θ[4]) ├─┤ Rz(θ[5]) ├─────────┤ X ├──────────■──────»
     ├──────────┤ ├──────────┤ ├──────────┤         └───┘        ┌─┴─┐    »
q_3: ┤ Ry(x[3]) ├─┤ Ry(θ[6]) ├─┤ Rz(θ[7]) ├──────────────────────┤ X ├────»
     ├──────────┤ ├──────────┤ ├──────────┤                      └───┘    »
q_4: ┤ Ry(x[4]) ├─┤ Ry(θ[8]) ├─┤ Rz(θ[9]) ├───────────────────────────────»
     ├──────────┤┌┴──────────┤┌┴──────────┤                               »
q_5: ┤ Ry(x[5]) ├┤ Ry(θ[10]) ├┤ Rz(θ[11]) ├───────────────────────────────»
     └──────────┘└───────────┘└───────────┘                               »
«           

In [18]:
observable = SparsePauliOp(
    "ZZZZZZ"
)

vqr = VQR(
    feature_map=feature_map,
    ansatz=ansatz,
    observable=observable,
    loss="squared_error",
    optimizer=COBYLA(maxiter=100)
)

In [19]:
print("Starting VQR training...")

vqr.fit(
    X_train,
    y_train
)

print("Training complete!")

Starting VQR training...
Training complete!


In [20]:
y_pred = vqr.predict(X_test)

y_pred = np.asarray(y_pred).flatten()

print("First 10 predictions:")
print(y_pred[:10])

print("\nFirst 10 actual values:")
print(y_test[:10])

First 10 predictions:
[0.58691101 0.96469677 0.66792202 0.97507403 0.70057766 0.95848746
 0.97435686 0.98625114 0.95889631 0.95792621]

First 10 actual values:
[10. 15.  2. 13. 20. 17. 19. 18. 29. 28.]


In [21]:
r2 = r2_score(
    y_test,
    y_pred
)

print("R²:", r2)

R²: -2.0710953875823437


In [22]:
rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

print("RMSE:", rmse)

RMSE: 15.553029571846455


In [23]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

print("MAE:", mae)

MAE: 12.848148470349038


In [24]:
print("=" * 40)
print("VQR RESULTS")
print("=" * 40)

print(f"R²   : {r2:.6f}")
print(f"RMSE : {rmse:.6f} m")
print(f"MAE  : {mae:.6f} m")

VQR RESULTS
R²   : -2.071095
RMSE : 15.553030 m
MAE  : 12.848148 m
